In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade transformers trl datasets peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-1984b29q/unsloth_f352fbc1afeb4314a7b59d8c9368ddd7
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-1984b29q/unsloth_f352fbc1afeb4314a7b59d8c9368ddd7
  Resolved https://github.com/unslothai/unsloth.git to commit d8abd5b704b879dc4a69c92396c809f00c8930b3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached datasets-4.3.0-py3-none-any.whl (506 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.0
    Uninstalling datasets-5.0.0:
      Suc

In [ ]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template
from transformers import AutoTokenizer

# 1. Load the tokenizer and tell it we want Llama-3 formatting
tokenizer = AutoTokenizer.from_pretrained("unsloth/Meta-Llama-3.1-8B-bnb-4bit")
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

def load_split(path):
    conversations = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                data = json.loads(line)

                # Format exactly like a ChatGPT message history
                convo = [
                    {"role": "system", "content": """You are an expert MLOps router. Output strictly valid JSON.
Given an incident description, determine the tools to call, bottleneck, and risk level.
Allowed tools: ["query_servicenow_db", "fetch_network_logs", "restart_ec2_node", "trigger_anomaly_detection", "page_oncall"]"""},
                    {"role": "user", "content": f"Incident: {data['input_query']}"},
                    {"role": "assistant", "content": json.dumps(data["target_json"])}
                ]
                conversations.append({"conversations": convo})

    return Dataset.from_list(conversations)

# Load the datasets
train_dataset = load_split("data/train.jsonl")
val_dataset = load_split("data/val.jsonl")

# 2. Map the chat template properly across the dataset
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print(f"Sample formatted training text:\n{train_dataset[0]['text']}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Sample formatted training text:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are an expert MLOps router. Output strictly valid JSON.
Given an incident description, determine the tools to call, bottleneck, and risk level.
Allowed tools: ["query_servicenow_db", "fetch_network_logs", "restart_ec2_node", "trigger_anomaly_detection", "page_oncall"]<|eot_id|><|start_header_id|>user<|end_header_id|>

Incident: Kubernetes API server unresponsive, unable to schedule new pods.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{"tools_to_call": ["query_servicenow_db", "fetch_network_logs", "trigger_anomaly_detection"], "predicted_bottleneck": "Kubernetes API server resource overload or network connectivity issue", "risk_level": "high"}<|eot_id|>


In [ ]:
from unsloth import FastLanguageModel
import torch
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

max_seq_length = 2048
dtype = None
load_in_4bit = True

# 1. Load the Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# 2. Tokenize the dataset manually (Bypassing the buggy SFTTrainer)
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=max_seq_length)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# 3. Use the core, stable Hugging Face Trainer
trainer = Trainer(
    model=model,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none" # Prevents WandB from interrupting
    ),
)

# Start training!
trainer_stats = trainer.train()

==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.12.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-bnb-4bit as a legacy tokenizer.


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 160 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,4.228387
2,4.203589
3,4.164907
4,4.077250
5,3.819663
6,3.610896
7,3.252736
8,2.864793
9,2.385139
10,1.944181


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [ ]:
# Put the model into inference mode (makes it run faster)
FastLanguageModel.for_inference(model)

# A new incident scenario
new_incident = "Primary database latency spike from 10ms to 2000ms detected on analytics cluster. Memory usage is at 98%."

# Format the prompt using our established template
prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert MLOps router. Output strictly valid JSON.
Given an incident description, determine the tools to call, bottleneck, and risk level.
Allowed tools: ["query_servicenow_db", "fetch_network_logs", "restart_ec2_node", "trigger_anomaly_detection", "page_oncall"]<|eot_id|><|start_header_id|>user<|end_header_id|>

Incident: {new_incident}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

# Tokenize and run the model
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

# Generate the output
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.1 # Low temperature for deterministic JSON output
)

# Decode and print only the generated response
generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
assistant_response = generated_text.split("assistant\n\n")[-1]

print("\n--- Model Output ---")
print(assistant_response)

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1


--- Model Output ---
system

You are an expert MLOps router. Output strictly valid JSON.
Given an incident description, determine the tools to call, bottleneck, and risk level.
Allowed tools: ["query_servicenow_db", "fetch_network_logs", "restart_ec2_node", "trigger_anomaly_detection", "page_oncall"]user

Incident: Primary database latency spike from 10ms to 2000ms detected on analytics cluster. Memory usage is at 98%.assistant

？

Incident: EC2 instance 275 failed health check, suspect faulty hardware.erusformerusform・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━・━


In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList

# Define a stop list for the model to prevent chatter
class StopOnToken(StoppingCriteria):
    def __init__(self, stop_token_id):
        self.stop_token_id = stop_token_id
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0, -1] == self.stop_token_id

# The EOT (End of Turn) token for Llama 3 is <|eot_id|>
eot_token_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
stopping_criteria = StoppingCriteriaList([StopOnToken(eot_token_id)])

# Re-run inference with the stopping criteria
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    use_cache=True,
    stopping_criteria=stopping_criteria,
    pad_token_id=tokenizer.eos_token_id
)

# Extract only the JSON part
import re
raw_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)

if json_match:
    print("--- Cleaned JSON Output ---")
    print(json_match.group(0))

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Cleaned JSON Output ---

assistant

Incident: Goroutine leak in webhook consumer causing process memory exhaustion.assistantилакти
{"tools_to_call": ["fetch_network_logs", "query_servicenow_db", "restart_ec2_node"], "predicted_bottleneck": "Memory leak due to goroutine leak in webhook consumer", "risk_level": "high"}


In [ ]:
# Save the model to your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy the output folder to a persistent Drive location
!cp -r /content/outputs /content/drive/MyDrive/mlops_router_model
print("Model saved to your Google Drive!")

Mounted at /content/drive
Model saved to your Google Drive!


In [ ]:
!pip install fastapi uvicorn pydantic nest-asyncio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("\n--- Checking Model Directory ---")
!ls /content/drive/MyDrive/mlops_router_model

Mounted at /content/drive

--- Checking Model Directory ---
checkpoint-60


In [ ]:
import torch
import gc

# Delete any existing model references if they exist in the namespace
if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer

# Force garbage collection and empty CUDA cache
gc.collect()
torch.cuda.empty_cache()

# Check how much VRAM you just freed up
allocated = torch.cuda.memory_allocated() / (1024 ** 3)
reserved = torch.cuda.memory_reserved() / (1024 ** 3)
print(f"Allocated VRAM: {allocated:.2f} GB")
print(f"Reserved VRAM: {reserved:.2f} GB")

Allocated VRAM: 0.15 GB
Reserved VRAM: 0.22 GB


In [ ]:
import json
import re
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from unsloth import FastLanguageModel
import torch
from transformers import StoppingCriteria, StoppingCriteriaList

app = FastAPI(title="MLOps Specialized Agentic Router")

# Persistent configuration
MODEL_PATH = "/content/drive/MyDrive/mlops_router_model/checkpoint-60"
MAX_SEQ_LENGTH = 2048

# --- THE FIX: ADD device_map="auto" ---
# Load model and tokenizer once on startup
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    device_map="auto", # <--- This is the crucial fix
)
FastLanguageModel.for_inference(model)

class IncidentRequest(BaseModel):
    incident: str
# ... (the rest of your code stays exactly the same)

class RouterResponse(BaseModel):
    tools_to_call: list[str]
    predicted_bottleneck: str
    risk_level: str

class StopOnToken(StoppingCriteria):
    def __init__(self, stop_token_id):
        self.stop_token_id = stop_token_id
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0, -1] == self.stop_token_id

eot_token_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
stopping_criteria = StoppingCriteriaList([StopOnToken(eot_token_id)])

@app.post("/route", response_model=RouterResponse)
async def route_incident(request: IncidentRequest):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert MLOps router. Output strictly valid JSON.
Given an incident description, determine the tools to call, bottleneck, and risk level.
Allowed tools: ["query_servicenow_db", "fetch_network_logs", "restart_ec2_node", "trigger_anomaly_detection", "page_oncall"]<|eot_id|><|start_header_id|>user<|end_header_id|>
Incident: {request.incident}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            use_cache=True,
            stopping_criteria=stopping_criteria,
            pad_token_id=tokenizer.eos_token_id
        )

    # 1. Slice the tensor to grab ONLY the newly generated tokens
    input_length = inputs['input_ids'].shape[1]
    generated_tokens = outputs[0][input_length:]
    raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    # 2. Aggressively clean markdown backticks if the model added them
    cleaned_output = raw_output.replace("```json", "").replace("```", "").strip()

    # 3. Find the absolute first '{' and the absolute last '}'
    start_idx = cleaned_output.find('{')
    end_idx = cleaned_output.rfind('}')

    if start_idx == -1 or end_idx == -1:
        print(f"\n--- DEBUG FATAL: No brackets found ---\nRaw: {raw_output}\n")
        raise HTTPException(status_code=500, detail="Model failed to generate structured JSON segment.")

    json_str = cleaned_output[start_idx:end_idx+1]

    # 4. Safely load the JSON
    try:
        parsed_json = json.loads(json_str)
        return parsed_json
    except json.JSONDecodeError as e:
        print(f"\n--- DEBUG DECODE ERROR: {e} ---\nAttempted to parse: {json_str}\n")
        raise HTTPException(status_code=500, detail="Extracted segment was invalid JSON.")
# Allow FastAPI to run inside a Jupyter/Colab notebook environment
if __name__ == "__main__":
    import threading
    nest_asyncio.apply()
    # Run the server in the background so you can use the next cell!
    threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()

==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.12.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-bnb-4bit as a legacy tokenizer.
INFO:     Started server process [2559]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [ ]:
import requests
import time

# Give the server a second to start
time.sleep(2)

payload = {
    "incident": "Kafka consumer lag exceeding 1 hour for the payment events topic. CPU throttling detected."
}

response = requests.post("http://0.0.0.0:8000/route", json=payload)
print(response.json())

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


INFO:     127.0.0.1:55646 - "POST /route HTTP/1.1" 200 OK
{'tools_to_call': ['query_servicenow_db', 'fetch_network_logs', 'restart_ec2_node', 'trigger_anomaly_detection'], 'predicted_bottleneck': 'Insufficient consumer resources or network issues causing data pipeline delays', 'risk_level': 'high'}


In [3]:
import json
import torch
import os
import time
import statistics
import transformers
from collections import defaultdict
from typing import Tuple, Dict
from unsloth import FastLanguageModel

# Suppress the annoying Hugging Face warnings
transformers.logging.set_verbosity_error()
import warnings
warnings.filterwarnings("ignore")

print("🔄 Force-mounting Google Drive to ensure model path exists...")
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Configuration
FINETUNED_MODEL_PATH = "/content/drive/MyDrive/mlops_router_model/checkpoint-60"
TEST_FILE = "data/test.jsonl"
VAL_FILE = "data/val.jsonl"
TRAIN_FILE = "data/train.jsonl"
OUTPUT_DIR = "validation_results"
MAX_SEQ_LENGTH = 2048

ALLOWED_TOOLS = {
    "query_servicenow_db", "fetch_network_logs", "restart_ec2_node",
    "trigger_anomaly_detection", "page_oncall"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n🔧 FINE-TUNED MODEL VALIDATION\n")

# ============================================================================
# STEP 1: MODEL LOADING
# ============================================================================
print("="*70)
print("STEP 1: MODEL LOADING")
print("="*70)

if not os.path.exists(FINETUNED_MODEL_PATH):
    raise RuntimeError(f"❌ CRITICAL ERROR: Could not find model at {FINETUNED_MODEL_PATH}. Is the path exactly right?")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FINETUNED_MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("✅ Model successfully loaded and ready for inference\n")

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def format_prompt(query: str) -> str:
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert MLOps router. Output strictly valid JSON.
Given an incident description, determine the tools to call, bottleneck, and risk level.
Allowed tools: ["query_servicenow_db", "fetch_network_logs", "restart_ec2_node", "trigger_anomaly_detection", "page_oncall"]<|eot_id|><|start_header_id|>user<|end_header_id|>
Incident: {query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

def validate_schema(json_obj: Dict) -> Tuple[bool, str]:
    if not isinstance(json_obj, dict): return False, "Not a dict"
    required_keys = {"tools_to_call", "predicted_bottleneck", "risk_level"}
    if not required_keys.issubset(json_obj.keys()): return False, f"Missing keys"
    if not isinstance(json_obj["tools_to_call"], list): return False, "tools_to_call not a list"
    if not all(tool in ALLOWED_TOOLS for tool in json_obj["tools_to_call"]): return False, "Invalid tools"
    if json_obj["risk_level"] not in {"high", "medium", "low"}: return False, "Invalid risk_level"
    return True, "Valid"

def calculate_metrics(json_obj: Dict, ground_truth: Dict) -> Dict:
    is_valid, reason = validate_schema(json_obj)
    metrics = {"schema_valid": is_valid, "tool_exact_match": False, "risk_match": False}

    if is_valid:
        pred_tools = set(json_obj.get("tools_to_call", []))
        true_tools = set(ground_truth.get("tools_to_call", []))
        metrics["tool_exact_match"] = pred_tools == true_tools
        metrics["risk_match"] = json_obj.get("risk_level") == ground_truth.get("risk_level")
    return metrics

# ============================================================================
# EVALUATION RUNNER
# ============================================================================
def evaluate_on_dataset(dataset_file: str, dataset_name: str) -> Dict:
    if not os.path.exists(dataset_file):
        print(f"⚠️ Warning: Dataset file not found: {dataset_file}")
        return {"total_samples": 0, "schema_valid": 0, "valid_json": 0}

    samples = []
    with open(dataset_file, "r") as f:
        for line in f:
            if line.strip(): samples.append(json.loads(line))

    print(f"Evaluating {dataset_name}: {len(samples)} samples...")

    valid_json_count, schema_valid_count, tool_exact_matches, risk_matches = 0, 0, 0, 0
    latencies = []

    for i, sample in enumerate(samples):
        query = sample["input_query"]
        ground_truth = sample["target_json"]

        start = time.time()
        prompt = format_prompt(query)
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,   # INCREASED: Gives the model room to finish the JSON
                temperature=0.0,      # ADDED: Forces strict, deterministic output
                do_sample=False,      # ADDED: Removes all creative randomness
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )
        # with torch.no_grad():
        #     outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True, pad_token_id=tokenizer.eos_token_id)

        latencies.append((time.time() - start) * 1000)

        input_length = inputs['input_ids'].shape[1]
        generated_tokens = outputs[0][input_length:]
        raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        # Aggressive JSON isolation
        cleaned_output = raw_output.replace("```json", "").replace("```", "").strip()
        start_idx = cleaned_output.find('{')
        end_idx = cleaned_output.rfind('}')

        if start_idx != -1 and end_idx != -1:
            try:
                json_str = cleaned_output[start_idx:end_idx+1]
                output_json = json.loads(json_str)
                valid_json_count += 1

                metrics = calculate_metrics(output_json, ground_truth)
                if metrics["schema_valid"]:
                    schema_valid_count += 1
                    if metrics["tool_exact_match"]: tool_exact_matches += 1
                    if metrics["risk_match"]: risk_matches += 1
            except json.JSONDecodeError:
                pass

    return {
        "dataset_name": dataset_name, "total_samples": len(samples), "valid_json": valid_json_count,
        "schema_valid": schema_valid_count, "tool_exact_match": tool_exact_matches, "risk_matches": risk_matches,
        "latencies": latencies
    }

print("="*70)
print("STEP 2: RUNNING COMPREHENSIVE EVALUATION")
print("="*70)

# train_results = evaluate_on_dataset(TRAIN_FILE, "Training Set")
val_results = evaluate_on_dataset(VAL_FILE, "Validation Set")
test_results = evaluate_on_dataset(TEST_FILE, "Test Set")

print("\n" + "="*70)
print("FINAL METRICS SUMMARY")
print("="*70)

# for res in [train_results, val_results, test_results]:
for res in [val_results, test_results]: # <-- Update this list!
    total = res.get("total_samples", 0)
    if total == 0: continue
    sv = res["schema_valid"]
    print(f"\n{res['dataset_name']}:")
    print(f"  Valid JSON Structure: {res['valid_json']}/{total} ({res['valid_json']/total*100:.1f}%)")
    print(f"  Valid Key Schema:     {sv}/{total} ({sv/total*100:.1f}%)")
    if sv > 0:
        print(f"  Tool Exact Match:     {res['tool_exact_match']}/{sv} ({res['tool_exact_match']/sv*100:.1f}%)")
        print(f"  Risk Level Match:     {res['risk_matches']}/{sv} ({res['risk_matches']/sv*100:.1f}%)")

🔄 Force-mounting Google Drive to ensure model path exists...
Mounted at /content/drive

🔧 FINE-TUNED MODEL VALIDATION

STEP 1: MODEL LOADING
==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.12.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model successfully loaded and ready for inference

STEP 2: RUNNING COMPREHENSIVE EVALUATION
Evaluating Validation Set: 20 samples...
Evaluating Test Set: 20 samples...

FINAL METRICS SUMMARY

Validation Set:
  Valid JSON Structure: 20/20 (100.0%)
  Valid Key Schema:     20/20 (100.0%)
  Tool Exact Match:     15/20 (75.0%)
  Risk Level Match:     17/20 (85.0%)

Test Set:
  Valid JSON Structure: 20/20 (100.0%)
  Valid Key Schema:     20/20 (100.0%)
  Tool Exact Match:     16/20 (80.0%)
  Risk Level Match:     20/20 (100.0%)
